# Module 5 — Inverse Kinematics
## Unit 2 — Inverse Kinematics of One and Two Joints
### Lesson 2.3 — Elbow-Up and Elbow-Down: the Two Solutions

*Physical AI Curriculum · hands-on notebook. Run **Kernel → Restart & Run All**.*


## Demonstration (§7)
Both solutions for one target; verify each by forward kinematics.


In [ ]:
import numpy as np
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt

def fk_two_link(theta1, theta2, L1, L2):
    """Forward kinematics: planar 2-link gripper position."""
    x = L1*np.cos(theta1) + L2*np.cos(theta1+theta2)
    y = L1*np.sin(theta1) + L2*np.sin(theta1+theta2)
    return np.array([x, y])

def reachable(x, y, L1, L2, tol=1e-9):
    r = np.hypot(x, y)
    return abs(L1-L2)-tol <= r <= L1+L2+tol

def ik_two_link(x, y, L1, L2):
    """Return both (theta1,theta2) solutions; [] if unreachable, one if on boundary."""
    r2 = x*x + y*y
    c2 = (r2 - L1*L1 - L2*L2)/(2*L1*L2)
    if c2 < -1-1e-9 or c2 > 1+1e-9:
        return []
    c2 = max(-1.0, min(1.0, c2))
    sols=[]
    for sign in (+1.0, -1.0):
        t2 = sign*np.arccos(c2)
        t1 = np.arctan2(y, x) - np.arctan2(L2*np.sin(t2), L1+L2*np.cos(t2))
        sols.append((t1, t2))
        if abs(np.sin(t2)) < 1e-6:    # boundary: the two coincide
            break
    return sols

L1,L2=0.4,0.3
target=(0.5,0.0)
for (t1,t2) in ik_two_link(*target,L1,L2):
    pos=fk_two_link(t1,t2,L1,L2)
    print("theta1,theta2 (deg)=",np.round(np.degrees([t1,t2]),2)," -> gripper",np.round(pos,3))


## Coding Exercise (§8)
Return both solutions; assert each lands on the target.


In [ ]:
# YOUR CODE HERE
sols=ik_two_link(0.5,0.0,L1,L2)
assert len(sols)==2
for (t1,t2) in sols:
    assert np.allclose(fk_two_link(t1,t2,L1,L2),[0.5,0.0],atol=1e-9)
# the two elbow signs are opposite
assert sols[0][1]*sols[1][1] < 0
print("two solutions OK")


## Check your work


In [ ]:
# at the outer boundary the two merge into one
b=ik_two_link(0.7,0.0,L1,L2)
assert len(b)==1
print("All checks passed.")
